In [ ]:
# Standard library imports
import sys
from pathlib import Path
from datetime import datetime
from collections import deque

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from time import sleep

# Add parent directory to path to import from src/
parent_dir = Path().resolve().parent
sys.path.insert(0, str(parent_dir))

# Third-party imports
import numpy as np
import pandas as pd
from lightgbm import LGBMRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from xgboost import XGBRegressor
import plotly.express as px
import joblib

# Local imports
from src.cpu_temp_bundled import HardwareMonitor

# Background color white
px.defaults.template = "plotly_white"

In [ ]:
class ComputerInfoExtractor:
    """
    Handles data extraction and preprocessing for CPU temperature monitoring.
    Separates data collection concerns from model training.
    """

    def __init__(self, scaler_type='standard', use_time_features=True, lag_steps = [1, 2, 4, 8, 12], rolling_windows = [6, 12, 24, 36]):
        self.data = pd.DataFrame()
        self.pc_info = {}
        self.scaler_type = scaler_type
        self.use_time_features = use_time_features
        self.lag_steps = lag_steps
        self.rolling_windows = rolling_windows

    def extract_CPU_data(self, mean_time=None, csv=False):
        """
        Load CPU data for training from CSV.

        Args:
            mean_time: Optional time window (seconds) for resampling and averaging data. None = no resampling
            csv: If True, load from '../data/new_latest.csv'
        """
        if csv:
            self.data = pd.read_csv('../data/new_latest.csv')
            if 'timestamp' in self.data.columns:
                self.data['timestamp'] = pd.to_datetime(self.data['timestamp'])

        if mean_time not in (None, 0):
            self.data = self.data.resample(f"{mean_time}s", on='timestamp').mean().reset_index()
            # Recreate sequential time column after resampling
            if 'time' in self.data.columns:
                self.data['time'] = range(len(self.data))

        return self.data

    def extract_PC_info(self):
        """Extract PC hardware information."""
        with HardwareMonitor() as monitor:
            info = monitor.get_hardware_info()
            pc_dict = {}
            # Access information
            if info['cpu']:
                pc_dict['CPU'] = info['cpu']['name']

            if info['gpu']:
                pc_dict['GPU'] = info['gpu']['name']

            if info['motherboard']:
                pc_dict['Motherboard'] = info['motherboard']['name']

            if info['ram']:
                pc_dict['RAM'] = info['ram']['name']

            for disk in info['storage']:
                pc_dict["Storage"] = disk['name']

            self.pc_info = pc_dict
            return pc_dict

    def plot_CPU_data(self):
        """Plot CPU data with Plotly."""
        if self.pc_info == {}:
            self.extract_PC_info()
        px.line(self.data, x='timestamp', y=['cpu_temp','cpu_load','cpu_power','cpu_clock','cpu_volt','gpu_temp','gpu_load','gpu_power','mb_temp','ram_load','fan_rpm'], title=f'{self.pc_info.get("CPU")} Temperature').show()

    def create_time_features_on_df(self):
        """Create lag, rolling, and diff features for better predictions."""

        if self.data.empty:
            self.extract_CPU_data()

        exclude_cols = ['cpu_temp', 'time', 'fan_rpm', 'timestamp']
        feature_cols = [col for col in self.data.columns if col not in exclude_cols and self.data[col].dtype in ['float64', 'int64', 'float32', 'int32']]

        new_features = {}

        for col in feature_cols:
            # Lag features
            for lag in self.lag_steps:
                new_features[f'{col}_lag_{lag}'] = self.data[col].shift(lag)

            # Rolling statistics
            for window in self.rolling_windows:
                new_features[f'{col}_rolling_mean_{window}'] = self.data[col].rolling(window=window).mean()
                new_features[f'{col}_rolling_std_{window}'] = self.data[col].rolling(window=window).std()

            # Rate of change
            new_features[f'{col}_diff_1'] = self.data[col].diff(1)

        if new_features:
            new_cols_df = pd.DataFrame(new_features, index=self.data.index)
            df = pd.concat([self.data, new_cols_df], axis=1)
        else:
            df = self.data.copy()

        return df

    def extract_data_pipeline(self, csv=False, mean_time=None):
        """Full data extraction and preprocessing pipeline."""
        if self.data.empty:
            self.extract_CPU_data(mean_time=mean_time, csv=csv)

        if self.use_time_features:
            df = self.create_time_features_on_df()
        else:
            df = self.data.copy()

        # Fill missing values but DO NOT scale here to avoid data leakage
        # Scaling will be done in CoreTempRegressor.fit_predict() after train/test split
        self.processed_data = df.ffill().bfill().reset_index(drop=True)
        return self.processed_data

In [ ]:
class CoreTempRegressor:
    """
    CPU Temperature anomaly detection using regression models.
    Receives pre-processed data from ComputerInfoExtractor.
    Can be used for training, saving/loading models, and real-time anomaly detection.

    IMPORTANT: Scaling is performed AFTER train/test split to prevent data leakage.
    """

    PLOT_STYLE = {
        'line_width': 3,
        'marker_size': 8,
        'threshold_line_width': 5,
        'threshold_opacity': 0.5,
        'error_color': 'orange',
        'lower_threshold_color': 'green',
        'upper_threshold_color': 'red',
        'line_dash': 'dash'
    }

    def __init__(self, extractor=None):
        self.extractor = extractor
        self.scaler = None
        self.model = None
        self.data = None
        self.predict_data = None

        # Store for predictions
        self.x_train = None
        self.y_train = None
        self.x_test = None
        self.y_test = None
        self.y_pred = None
        self.test_data = None

        # Store feature engineering parameters
        self.use_time_features = None

        # Store model name
        self.model_name = None
        self.multi_variable = True

        # Anomaly detection thresholds
        self.low_threshold = None
        self.high_threshold = None
        self.threshold_std = 1.5

        # Real-time monitoring buffer
        self.data_buffer = None
        self.time_counter = 0

        # Feature columns (set after training)
        self.feature_columns = None

    def set_data(self, data: pd.DataFrame):
        """Set pre-processed data from ComputerInfoExtractor."""
        self.data = data.copy()
        return self

    def configure_model(self, model='linear', multi_variable=True, scaler='standard', use_time_features=True):
        """Configure the model and scaler to use."""
        scaler_dict = {
            'standard': StandardScaler(),
            'minmax': MinMaxScaler(),
            'robust': RobustScaler()
        }

        model_dict = {
            'xgb': XGBRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_weight=10,
                reg_alpha=0.5,
                reg_lambda=2.0,
                random_state=42,
                tree_method='hist'
            ),
            'linear': Ridge(
                alpha=10.0,
                solver='auto',
                random_state=42
            ),
            'lightgbm': LGBMRegressor(
                n_estimators=300,
                max_depth=6,
                learning_rate=0.05,
                num_leaves=31,
                subsample=0.8,
                colsample_bytree=0.8,
                min_child_samples=100,
                reg_alpha=0.5,
                reg_lambda=2.0,
                random_state=42,
                verbose=-1
            ),
        }

        self.model = model_dict.get(model)
        self.scaler = scaler_dict.get(scaler)
        self.model_name = model
        self.use_time_features = use_time_features
        self.multi_variable = multi_variable

    def fit_predict(self, train_size=0.8, threshold_std=1.5):
        """
        Train the model and make predictions on test set.
        Data should already be pre-processed by ComputerInfoExtractor.

        CRITICAL: Scaling is done HERE after train/test split to prevent data leakage.
        The scaler is fit ONLY on training data, then applied to test data.
        """
        if self.data is None:
            raise ValueError("No data set. Use set_data() with pre-processed data from ComputerInfoExtractor.")

        self.threshold_std = threshold_std

        # Data is already processed (features created, missing values filled)
        df = self.data.copy()

        # Split the data BEFORE scaling
        split_idx = int(len(df) * train_size)
        train_data = df[:split_idx].copy()
        test_data = df[split_idx:].copy()

        # Drop NaN rows (from lag/rolling features)
        train_data = train_data.dropna()
        test_data = test_data.dropna()

        # Prepare X and y
        exclude_cols = ['cpu_temp', 'time', 'fan_rpm', 'timestamp']
        if self.multi_variable:
            feature_cols = [col for col in train_data.columns if col not in exclude_cols]
            self.x_train = train_data[feature_cols]
            self.y_train = train_data['cpu_temp']

            self.x_test = test_data[feature_cols]
            self.y_test = test_data['cpu_temp']
        else:
            self.x_train = train_data[['time']]
            self.y_train = train_data['cpu_temp']

            self.x_test = test_data[['time']]
            self.y_test = test_data['cpu_temp']

        # Store feature columns for later use
        self.feature_columns = list(self.x_train.columns)

        # Fit scaler ONLY on training data (prevents data leakage)
        x_train_scaled = self.scaler.fit_transform(self.x_train)
        x_test_scaled = self.scaler.transform(self.x_test)

        # Train model
        self.model.fit(x_train_scaled, self.y_train)

        # Make predictions
        self.y_pred = self.model.predict(x_test_scaled)

        # Store test_data for plotting
        self.test_data = test_data

        # Calculate anomaly thresholds
        diff = self.y_test.values - self.y_pred
        mean_diff = np.mean(diff)
        std_diff = np.std(diff)
        self.low_threshold = mean_diff - threshold_std * std_diff
        self.high_threshold = mean_diff + threshold_std * std_diff

    def predict(self, X):
        """Make predictions on new data."""
        X_scaled = self.scaler.transform(X)
        self.predict_data = self.model.predict(X_scaled)
        return self.predict_data

    def get_thresholds(self):
        """Return current anomaly thresholds."""
        return {
            'low': self.low_threshold,
            'high': self.high_threshold,
            'std_multiplier': self.threshold_std
        }

    def recalculate_thresholds(self, threshold_std: float):
        """Recalculate thresholds with new std multiplier without retraining."""
        if self.y_test is None or self.y_pred is None:
            raise ValueError("Model must be trained before recalculating thresholds")

        self.threshold_std = threshold_std
        diff = self.y_test.values - self.y_pred
        mean_diff = np.mean(diff)
        std_diff = np.std(diff)
        self.low_threshold = mean_diff - threshold_std * std_diff
        self.high_threshold = mean_diff + threshold_std * std_diff

    def init_realtime_buffer(self):
        """Initialize the buffer for real-time anomaly detection."""
        if self.extractor is None:
            raise ValueError("Extractor required for real-time detection. Pass ComputerInfoExtractor to constructor.")
        max_window = max(self.extractor.rolling_windows) if self.extractor.rolling_windows else 7
        max_lag = max(self.extractor.lag_steps) if self.extractor.lag_steps else 3
        buffer_size = max(max_window, max_lag) + 5
        self.data_buffer = deque(maxlen=buffer_size)
        self.time_counter = 0

    def detect_anomaly(self, current_data: dict) -> tuple:
        """
        Detect if current data point is an anomaly.
        Uses extractor to create time features from buffer.

        Args:
            current_data: Dict with sensor readings from HardwareMonitor.get_essential_fast()

        Returns:
            tuple: (is_anomaly: bool, diff: float, predicted: float, actual: float)
        """
        if self.extractor is None:
            raise ValueError("Extractor required for real-time detection. Pass ComputerInfoExtractor to constructor.")

        if self.data_buffer is None:
            self.init_realtime_buffer()

        # Add timestamp and time
        current_data = current_data.copy()
        current_data['timestamp'] = datetime.now()
        current_data['time'] = self.time_counter
        self.time_counter += 1

        # Add to buffer
        self.data_buffer.append(current_data)

        # Need enough history for features
        min_required = max(max(self.extractor.rolling_windows), max(self.extractor.lag_steps)) + 1
        if len(self.data_buffer) < min_required:
            return False, 0.0, 0.0, current_data.get('cpu_temp', 0.0)

        # Create DataFrame from buffer and use extractor for feature engineering
        self.extractor.data = pd.DataFrame(list(self.data_buffer))

        if self.use_time_features and self.multi_variable:
            df = self.extractor.create_time_features_on_df()
            df = df.dropna()
        else:
            df = self.extractor.data.copy()

        if len(df) == 0:
            return False, 0.0, 0.0, current_data.get('cpu_temp', 0.0)

        # Get last row for prediction
        last_row = df.iloc[[-1]]

        # Prepare features
        if self.multi_variable:
            X = last_row.drop(['cpu_temp', 'time', 'fan_rpm', 'timestamp'], axis=1, errors='ignore')
        else:
            X = last_row[['time']]

        # Ensure columns match training
        if self.feature_columns:
            missing_cols = set(self.feature_columns) - set(X.columns)
            for col in missing_cols:
                X[col] = 0
            X = X[self.feature_columns]

        # Predict
        predicted = self.predict(X)[0]
        actual = current_data.get('cpu_temp', 0.0)
        diff = actual - predicted

        # Check anomaly
        is_anomaly = diff < self.low_threshold or diff > self.high_threshold

        return is_anomaly, diff, predicted, actual

    def plot_predictions(self):
        """Plot predictions and anomaly detection results."""
        # Use stored test_data time values
        time_test = self.test_data['time'].values
        timestamp_test = self.test_data['timestamp'].values

        # Get model name for titles
        model_display_name = self.model_name.upper() if self.model_name else "Model"

        # Prepare data
        df_test = self.x_test.copy()
        df_test['time'] = time_test
        df_test['timestamp'] = timestamp_test
        df_test["Actual"] = self.y_test.values
        df_test["Predicted"] = self.y_pred
        df_test = df_test.sort_values("time")

        df_test["diff"] = df_test["Actual"] - df_test["Predicted"]
        mean_diff = df_test["diff"].mean()
        stardard_dev = df_test["diff"].std()
        low_lim = mean_diff - self.threshold_std * stardard_dev
        high_lim = mean_diff + self.threshold_std * stardard_dev

        # Transform to long format (better for px.line)
        df_plot = df_test.melt(id_vars="timestamp", value_vars=["diff"], var_name="Type", value_name="Temperature Diff")

        # Predictions plot
        fig_pred = px.line(df_test, x="timestamp", y=["Actual", "Predicted"], title=f"{model_display_name} Regression: CPU Temp: Predicted vs Actual")
        fig_pred.update_traces(
            line_width=self.PLOT_STYLE['line_width'],
            marker={'size': self.PLOT_STYLE['marker_size']}
        )
        fig_pred.show()

        # Error plot
        fig = px.line(df_plot, x="timestamp", y="Temperature Diff", color="Type", title=f"{model_display_name} Regression: CPU Temp Difference: Predicted vs Actual")
        fig.update_traces(
            line_color=self.PLOT_STYLE['error_color'],
            line_width=self.PLOT_STYLE['line_width'],
            marker={'size': self.PLOT_STYLE['marker_size']}
        )
        fig.add_hline(
            y=low_lim,
            line_width=self.PLOT_STYLE['threshold_line_width'],
            line_color=self.PLOT_STYLE['lower_threshold_color'],
            line_dash=self.PLOT_STYLE['line_dash'],
            opacity=self.PLOT_STYLE['threshold_opacity']
        )
        fig.add_hline(
            y=high_lim,
            line_width=self.PLOT_STYLE['threshold_line_width'],
            line_color=self.PLOT_STYLE['upper_threshold_color'],
            line_dash=self.PLOT_STYLE['line_dash'],
            opacity=self.PLOT_STYLE['threshold_opacity']
        )
        fig.show()

        # Anomaly detection plot
        df_test["anomaly"] = (df_test["diff"] > high_lim) | (df_test["diff"] < low_lim)
        fig_anom = px.line(df_test, x="timestamp", y="anomaly", color="anomaly", title=f"{model_display_name} Regression: Anomaly Detection", markers=True)
        fig_anom.show()

    def evaluate_metrics(self):
        """Calculate and return evaluation metrics."""
        rmse = np.sqrt(mean_squared_error(self.y_test, self.y_pred))
        mape = np.mean(np.abs((self.y_test - self.y_pred) / self.y_test)) * 100
        r2 = r2_score(self.y_test, self.y_pred)
        mae = mean_absolute_error(self.y_test, self.y_pred)

        return {
            'rmse': rmse,
            'mape': mape,
            'r2': r2,
            'mae': mae
        }

    def save_model(self, path: str):
        """Save the trained model to file."""
        joblib.dump(self, path)

    @staticmethod
    def load_model(path: str) -> 'CoreTempRegressor':
        """Load a trained model from file."""
        from src.data_extractor import ComputerInfoExtractor

        model = joblib.load(path)

        # Backward compatibility: Add extractor if not present (for old models)
        if not hasattr(model, 'extractor') or model.extractor is None:
            # Create extractor with default parameters
            extractor = ComputerInfoExtractor(
                scaler_type='standard',
                use_time_features=getattr(model, 'use_time_features', True),
                lag_steps=getattr(model, 'lag_steps', [1, 2, 4, 8, 12]),
                rolling_windows=getattr(model, 'rolling_windows', [6, 12, 24, 36])
            )
            model.extractor = extractor

        # Initialize buffer for real-time detection if extractor exists
        if model.extractor and hasattr(model, 'init_realtime_buffer'):
            try:
                model.init_realtime_buffer()
            except:
                pass  # Buffer initialization might fail if extractor params are missing

        return model

In [ ]:
class ConvAutoencoder(nn.Module):

    def __init__(
        self,
        input_dim=7,
        encoder_channels=[64, 32, 16],
        decoder_channels=[16, 32, 64],
        window_size=60,
        epochs=20,
        batch_size=32,
        learning_rate=1e-4,
        test_size=0.2,
        threshold_std=2,
        *args, **kwargs
    ):
        super().__init__(*args, **kwargs)

        self.input_dim = input_dim
        self.window_size = window_size
        self.epochs = epochs
        self.batch_size = batch_size
        self.learning_rate = learning_rate
        self.test_size = test_size
        self.threshold_std = threshold_std

        self.data = None
        self.train_data = None
        self.train_loader = None
        self.test_data = None
        self.test_loader = None
        self.scaler = MinMaxScaler()
        self.feature_columns = None
        self.erros = None
        self.threshold = None

        encoder_layers = []
        encoder_current = input_dim

        for out_channels in encoder_channels:
            encoder_layers.append(nn.Conv1d(encoder_current, out_channels, kernel_size=3, stride=1, padding=1))
            encoder_layers.append(nn.ReLU())
            encoder_current = out_channels

        self.encoder = nn.Sequential(*encoder_layers)

        decoder_layers = []
        decoder_current = encoder_current

        for out_channels in decoder_channels:
            decoder_layers.append(nn.Conv1d(decoder_current, out_channels, kernel_size=3, stride=1, padding=1))
            decoder_layers.append(nn.ReLU())
            decoder_current = out_channels

        decoder_layers.append(nn.Conv1d(decoder_current, input_dim, kernel_size=3, stride=1, padding=1))
        decoder_layers.append(nn.Sigmoid())

        self.decoder = nn.Sequential(*decoder_layers)

    def make_windows(self, data):
        return torch.tensor(
            np.array([data[i:i+self.window_size] for i in range(len(data) - self.window_size)]),
            dtype=torch.float32
        )

    def process_data(self, processed_data):
        self.data = processed_data

        colunas_ae = ['cpu_temp', 'cpu_load', 'cpu_power', 'gpu_temp', 'gpu_load', 'gpu_power', 'ram_load']
        self.feature_columns = colunas_ae

        data = processed_data[colunas_ae].values
        data = self.scaler.fit_transform(data)

        windows = self.make_windows(data)

        train_data, test_data = train_test_split(windows, test_size=self.test_size, shuffle=False)
        self.train_data = train_data
        self.test_data = test_data

    def _get_test_timestamps(self, use_window_end=True):
        if self.data is None or self.test_data is None or 'timestamp' not in self.data.columns:
            return None

        n_windows = max(len(self.data) - self.window_size, 0)
        if n_windows == 0:
            return None

        split_idx = int((1 - self.test_size) * n_windows)
        offset = self.window_size - 1 if use_window_end else 0
        start_idx = split_idx + offset
        end_idx = start_idx + len(self.test_data)
        timestamps = self.data['timestamp'].iloc[start_idx:end_idx].reset_index(drop=True)

        if len(timestamps) != len(self.test_data):
            return None

        return pd.to_datetime(timestamps)

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x

    def fit(self):
        if self.train_data is None:
            print("No training data. Run process_data() first.")
            return

        train_dataset = torch.utils.data.TensorDataset(self.train_data)
        self.train_loader = DataLoader(train_dataset, batch_size=self.batch_size, shuffle=True)

        test_dataset = torch.utils.data.TensorDataset(self.test_data)
        self.test_loader = DataLoader(test_dataset, batch_size=self.batch_size, shuffle=False)

        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate)
        criterion = nn.MSELoss()

        train_losses = []
        test_losses = []

        for epoch in range(self.epochs):
            self.train()
            epoch_train_loss = 0

            for batch in self.train_loader:
                original_data = batch[0].permute(0, 2, 1)
                optimizer.zero_grad()
                reconstruction = self(original_data)
                loss = criterion(reconstruction, original_data)
                loss.backward()
                optimizer.step()
                epoch_train_loss += loss.item()

            train_losses.append(epoch_train_loss)

            self.eval()
            epoch_val_loss = 0

            with torch.no_grad():
                for batch in self.test_loader:
                    original_data = batch[0].permute(0, 2, 1)
                    reconstruction = self(original_data)
                    loss = criterion(reconstruction, original_data)
                    epoch_val_loss += loss.item()

            test_losses.append(epoch_val_loss)
            print(f"Epoch: {epoch+1}/{self.epochs} | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f}")

        return train_losses, test_losses

    def reconstruct(self):
        if self.test_loader is None:
            print("No test data. Run process_data() and fit() first.")
            return

        self.eval()
        with torch.no_grad():
            all_original = []
            all_reconstructed = []

            for batch in self.test_loader:
                x = batch[0].permute(0, 2, 1)
                output = self(x)
                all_original.append(x.permute(0, 2, 1).numpy())
                all_reconstructed.append(output.permute(0, 2, 1).numpy())

        all_original = np.concatenate(all_original, axis=0)
        all_reconstructed = np.concatenate(all_reconstructed, axis=0)

        original = all_original[:, 0, :]
        reconstruido = all_reconstructed[:, 0, :]

        original_denorm = self.scaler.inverse_transform(original)
        reconstruido_denorm = self.scaler.inverse_transform(reconstruido)

        df_orig = pd.DataFrame(original_denorm, columns=self.feature_columns)
        df_rec = pd.DataFrame(reconstruido_denorm, columns=[f'{col}_rec' for col in self.feature_columns])

        timestamp = self._get_test_timestamps(use_window_end=False)

        df = pd.concat([df_orig, df_rec], axis=1)
        if timestamp is not None:
            df['timestamp'] = timestamp
            x_col = 'timestamp'
        else:
            df['sample'] = range(len(df))
            x_col = 'sample'

        colunas_plot = self.feature_columns + [f'{col}_rec' for col in self.feature_columns]
        px.line(df, x=x_col, y=colunas_plot, title='Original vs Reconstruído').show()

    def reconstruction_error(self):
        if self.test_loader is None:
            print("No test data. Run process_data() and fit() first.")
            return

        self.eval()
        erros = []

        with torch.no_grad():
            for batch in self.test_loader:
                x = batch[0].permute(0, 2, 1)
                output = self(x)
                erro = ((output - x) ** 2).mean(dim=(1, 2))
                erros.extend(erro.numpy())

        erros = np.array(erros)
        threshold = erros.mean() + self.threshold_std * erros.std()

        return erros, threshold
    
    def fit_reconstruct(self, processed_data):
        self.process_data(processed_data)
        self.fit()
        self.reconstruct()
        erros, threshold = self.reconstruction_error()
        
        self.erros = erros
        self.threshold = threshold
    
    def plot_anomaly_detection(self):
        if self.erros is None or self.threshold is None:
            print("Run fit_reconstruct() first to calculate reconstruction errors and threshold.")
            self.erros, self.threshold = self.reconstruction_error()
            
        timestamp = self._get_test_timestamps(use_window_end=True)

        if timestamp is not None:
            df_plot = pd.DataFrame({'timestamp': timestamp, 'reconstruction_error': self.erros})
            fig = px.line(df_plot, x='timestamp', y='reconstruction_error', title='Reconstruction Error Over Time', color_discrete_sequence=['dimgray'])
        else:
            df_plot = pd.DataFrame({'sample': range(len(self.erros)), 'reconstruction_error': self.erros})
            fig = px.line(df_plot, x='sample', y='reconstruction_error', title='Reconstruction Error Over Time', color_discrete_sequence=['dimgray'])

        fig.add_hline(y=self.threshold, line_dash='dash', line_width=4, line_color='orange', annotation_text='Anomaly Threshold', annotation_position='top left')
        fig.update_traces(line=dict(width=3))
        fig.show()

In [ ]:
# Extract and prepare data using ComputerInfoExtractor
extractor = ComputerInfoExtractor(scaler_type = 'standard', use_time_features = False, lag_steps = [1, 2, 4, 8, 12], rolling_windows = [6, 12, 24, 36])
processed_data = extractor.extract_data_pipeline(csv = True, mean_time = None)
extractor.plot_CPU_data()

# Autoencoder

In [ ]:
model = ConvAutoencoder(input_dim=7, window_size=60, epochs=20, batch_size=32, learning_rate=1e-3, threshold_std=2)
model.fit_reconstruct(processed_data)
model.plot_anomaly_detection()

# Regressor

In [ ]:
# Instantiate regressor with extractor (for real-time detection) and set pre-processed data
core_predictor = CoreTempRegressor(extractor = extractor)
core_predictor.set_data(processed_data)

In [ ]:
# Training Linear model
core_predictor.configure_model(model="linear", scaler="standard", multi_variable=True, use_time_features=True)
core_predictor.fit_predict(train_size = 0.6, threshold_std = 3.0)
core_predictor.plot_predictions()

# Save the trained model
joblib.dump(core_predictor, '../models/cpu_temp_model_linear.joblib')

In [ ]:
# Training XGBoost model
core_predictor.configure_model(model = "xgb", scaler = "standard", multi_variable = True, use_time_features = True)
core_predictor.fit_predict(train_size = 0.6, threshold_std = 3.0)
core_predictor.plot_predictions()

# Save the trained model
joblib.dump(core_predictor, '../models/cpu_temp_model_xgb.joblib')

In [ ]:
# Training LightGBM model
core_predictor.configure_model(model = "lightgbm", scaler = "standard", multi_variable = True, use_time_features = True)
core_predictor.fit_predict(train_size = 0.6, threshold_std = 3.0)
core_predictor.plot_predictions()

# Save the trained model
joblib.dump(core_predictor, '../models/cpu_temp_model_lightgbm.joblib')